In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [31]:
import pandas as pd

file_path = '/content/drive/MyDrive/workshop8/trum_tweet_sentiment_analysis.csv'

trump_tweets_df = pd.read_csv(file_path)

display(trump_tweets_df.head())

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [25]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This...
  """
  # Convert the input to small/lower order.
  data = dataset.lower()
  # Remove URLs
  data = data # Placeholder for URL removal logic
  # Remove emojis
  data = data # Placeholder for emoji removal logic
  # Remove all other unwanted characters.
  data = data # Placeholder for unwanted character removal logic
  # Create tokens.
  tokens = data.split()
  # Remove stopwords:
  tokens = tokens # Placeholder for stopword removal logic
  if rule == "lemmatize":
    tokens = tokens # Placeholder for lemmatization logic
  elif rule == "stem":
    tokens = tokens # Placeholder for stemming logic
  else:
    print("Pick between lemmatize or stem")


  return " ".join(tokens)

In [26]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download necessary NLTK data (if not already downloaded)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [32]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function cleans text data by performing several preprocessing steps:
  - Lowercasing
  - Removing URLs
  - Removing emojis
  - Removing mentions, punctuation, and special characters
  - Tokenization
  - Removing stopwords
  - Optional lemmatization or stemming

  Args:
    dataset (str): The input text to be cleaned.
    rule (str): The rule for word normalization. Can be 'lemmatize' or 'stem'.

  Returns:
    str: The cleaned and processed text.
  """
  data = dataset.lower()

  data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)

  data = re.sub(r'@\w+', '', data)

  emoji_pattern = re.compile("[" # Start character set
                       "\U0001F600-\U0001F64F"  # emoticons
                       "\U0001F300-\U0001F5FF"  # symbols & pictographs
                       "\U0001F680-\U0001F6FF"  # transport & map symbols
                       "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                       "\U00002702-\U000027B0"
                       "\U000024C2-\U0001F251" # Dingbats etc.
                       "]+", flags=re.UNICODE)
  data = emoji_pattern.sub(r'', data)

  data = re.sub(f'[{re.escape(string.punctuation)}]', '', data)
  data = re.sub(r'[^\w\s]', '', data)
  data = re.sub(r'\d+', '', data)
  data = re.sub(r'\s+', ' ', data).strip()

  tokens = data.split()

  stop_words = set(stopwords.words('english'))
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)

In [34]:
trump_tweets_df['cleaned_text'] = trump_tweets_df['text'].apply(text_cleaning_pipeline)

display(trump_tweets_df.head())

,text,Sentiment,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,0,rt trump draining swamp taxpayer dollar trip a...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0,icymi hacker rig fm radio station play antitru...
2,Trump protests: LGBTQ rally in New York https:...,1,trump protest lgbtq rally new york bbcworld via
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0,hi im pier morgan david beckham awful donald t...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0,rt tech firm suing buzzfeed publishing unverif...


In [29]:
from sklearn.model_selection import train_test_split

X = trump_tweets_df['cleaned_text']
y = trump_tweets_df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1480098,)
X_test shape: (370025,)
y_train shape: (1480098,)
y_test shape: (370025,)


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Vectorization complete.")
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_tfidf, y_train)

print("Logistic Regression model training complete.")

y_pred = logistic_model.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

TF-IDF Vectorization complete.
Shape of X_train_tfidf: (1480098, 5000)
Shape of X_test_tfidf: (370025, 5000)
Logistic Regression model training complete.

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.91      0.87      0.89    121462

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025



# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.
